# **Data Cleaning, EDA, & Feature Engineering**

**Purpose**
- Prepare the merged CALABARZON dataset for analysis and modeling.

**Dataset**
- CALABARZON merged dengue dataset with climate, environmental, and socioeconomic variables.

**Process**
- Load required libraries.
- Review dataset structure.
- Perform data cleaning.
- Apply feature engineering.
- Conduct exploratory data analysis (descriptive statistics and correlation).
- Perform statistical tests (ANOVA and Chi-Square).

**Output**
- A cleaned dataset with engineered features and statistical insights for machine learning.

### **Libraries**

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import shapiro
from scipy.stats import f_oneway
from scipy.stats import chi2_contingency

### **Datset Overview**

In [ ]:
# Loading Dataset
df = pd.read_excel("Thesis_Dataset_Merged.xlsx")
df.head()

,Province,Municipality,Month,Year,Dengue Cases,Death Cases,Date,Rain Sum,Temperature,Relative Humidity,...,Private Infirmaries,Government Clinical Laboratories,Private Clinical Laboratories,Total Healthcare Facilities,Flood Hazard Level,Storm Surge Hazard Level,Total Population,Urban Population 2020,Urban Percent 2020,Density
0,CAVITE,ALFONSO,1,2020,4,0,2020-01-01,50.099998,22.956757,84.070856,...,0,0,2,4,LITTLE TO NONE,LITTLE TO NONE,59306.0,22203.0,37.438033,891.0
1,CAVITE,ALFONSO,2,2020,1,0,2020-02-01,42.500001,22.673062,79.557067,...,0,0,2,4,LITTLE TO NONE,LITTLE TO NONE,59306.0,22203.0,37.438033,891.0
2,CAVITE,ALFONSO,3,2020,0,0,2020-03-01,32.000000,24.933639,76.790074,...,0,0,2,4,LITTLE TO NONE,LITTLE TO NONE,59306.0,22203.0,37.438033,891.0
3,CAVITE,ALFONSO,4,2020,0,0,2020-04-01,86.300002,25.859916,77.456688,...,0,0,2,4,LITTLE TO NONE,LITTLE TO NONE,59306.0,22203.0,37.438033,891.0
4,CAVITE,ALFONSO,5,2020,0,0,2020-05-01,231.099997,26.513007,82.580199,...,0,0,2,4,LITTLE TO NONE,LITTLE TO NONE,59306.0,22203.0,37.438033,891.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8304 entries, 0 to 8303
Data columns (total 44 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Province                          8304 non-null   object        
 1   Municipality                      8304 non-null   object        
 2   Month                             8304 non-null   int64         
 3   Year                              8304 non-null   int64         
 4   Dengue Cases                      8304 non-null   int64         
 5   Death Cases                       8304 non-null   int64         
 6   Date                              8304 non-null   datetime64[ns]
 7   Rain Sum                          8304 non-null   float64       
 8   Temperature                       8304 non-null   float64       
 9   Relative Humidity                 8304 non-null   float64       
 10  Latitude                          8304 non-null 

### **Data Cleaning**

In [4]:
# Missing Values
df.isnull().sum()

Province                               0
Municipality                           0
Month                                  0
Year                                   0
Dengue Cases                           0
Death Cases                            0
Date                                   0
Rain Sum                               0
Temperature                            0
Relative Humidity                      0
Latitude                               0
Longitude                              0
Category                               0
Score                                  0
Total Percentage                       0
Provincial Score                       0
Annual Crop (%)                        0
Brush/Shrubs (%)                       0
Built-up (%)                           0
Closed Forest (%)                      0
Fishpond (%)                           0
Grassland (%)                          0
Inland Water (%)                       0
Mangrove Forest (%)                    0
Marshland/Swamp 

In [5]:
# Fixing Missing Values
df["Flood Hazard Level"] = df["Flood Hazard Level"].fillna(
    df["Flood Hazard Level"].mode()[0]
)

In [6]:
# Renaming Columns
df.rename(columns={"Rain Sum": "Rain",
                   "Category": "Municipal Class",
                   "Dengue_Incidence": "Dengue Incidence",
                   "Dengue_Risk_Level": "Dengue Risk Level"
}, inplace=True)

### **Feature Engineering**

In [7]:
# Settlement Type
df["Settlement Type"] = df["Urban Population 2020"].apply(
    lambda x: "URBAN" if x > 0 else "RURAL"
)

print(df[["Urban Population 2020", "Settlement Type"]].head())

   Urban Population 2020 Settlement Type
0                22203.0           URBAN
1                22203.0           URBAN
2                22203.0           URBAN
3                22203.0           URBAN
4                22203.0           URBAN


In [8]:
# Dengue Incidence
df["Dengue Incidence"] = (df["Dengue Cases"] / df["Total Population"]) * 1000
print(df[["Dengue Cases", "Total Population", "Dengue Incidence"]].head())

   Dengue Cases  Total Population  Dengue Incidence
0             4           59306.0          0.067447
1             1           59306.0          0.016862
2             0           59306.0          0.000000
3             0           59306.0          0.000000
4             0           59306.0          0.000000


In [9]:
# Dengue Risk Level
mean_incidence = df["Dengue Incidence"].mean()
std_incidence = df["Dengue Incidence"].std()

df["Dengue Risk Level"] = np.where(
    df["Dengue Incidence"] > mean_incidence + 1.5 * std_incidence,
    "High",
    np.where(
        df["Dengue Incidence"] > mean_incidence,
        "Medium",
        "Low"
    )
)

print(df[["Dengue Incidence", "Dengue Risk Level"]].head())

   Dengue Incidence Dengue Risk Level
0          0.067447               Low
1          0.016862               Low
2          0.000000               Low
3          0.000000               Low
4          0.000000               Low


In [10]:
# Summary statistics for Dengue Incidence
print(f"Mean: {mean_incidence:.2f}")
print(f"Standard Deviation: {std_incidence:.2f}")
print(df["Dengue Risk Level"].value_counts())

Mean: 0.11
Standard Deviation: 0.19
Dengue Risk Level
Low       5916
Medium    1898
High       490
Name: count, dtype: int64


In [11]:
# Sort by space and time
df = df.sort_values(["Province", "Municipality", "Year", "Month"])

# Lag period
lag = 1

# Lag dengue incidence with spaces
df[f"Dengue Incidence Lag {lag}"] = df.groupby(
    ["Province", "Municipality"]
)["Dengue Incidence"].shift(lag)

# Lag climate variables with spaces
climate_vars = ["Rain", "Temperature", "Relative Humidity"]
for var in climate_vars:
    df[f"{var} Lag {lag}"] = df.groupby(
        ["Province", "Municipality"]
    )[var].shift(lag)

# Drop rows with NaN (first month per municipality)
lagged_cols = [f"Dengue Incidence Lag {lag}"] + [f"{var} Lag {lag}" for var in climate_vars]
df = df.dropna(subset=lagged_cols)

In [12]:
# Aggregation of Land Cover 
df["Forest Cover (%)"] = (
    df["Closed Forest (%)"] +
    df["Open Forest (%)"] +
    df["Mangrove Forest (%)"]
)

df["Agricultural Land (%)"] = (
    df["Annual Crop (%)"] +
    df["Perennial Crop (%)"]
)

df["Water Bodies (%)"] = (
    df["Inland Water (%)"] +
    df["Fishpond (%)"] +
    df["Marshland/Swamp (%)"]
)

df["Open Vegetation (%)"] = (
    df["Grassland (%)"] +
    df["Brush/Shrubs (%)"]
)

df["Built-up / Barren (%)"] = (
    df["Built-up (%)"] +
    df["Open/Barren (%)"]
)

# Drop Original Land Cover Columns 
land_cover_cols = [
    "Annual Crop (%)", "Perennial Crop (%)",
    "Closed Forest (%)", "Open Forest (%)", "Mangrove Forest (%)",
    "Built-up (%)", "Open/Barren (%)",
    "Inland Water (%)", "Fishpond (%)", "Marshland/Swamp (%)",
    "Grassland (%)", "Brush/Shrubs (%)"
]

df.drop(columns=land_cover_cols, inplace=True)

In [13]:
# Aggregation of Healthcare Facilities 
df["Public Healthcare Facilities"] = (
    df["RHUs"] +
    df["Government Hospitals"] +
    df["Government PCFs"] +
    df["Government Infirmaries"] +
    df["Government Clinical Laboratories"]
)

df["Private Healthcare Facilities"] = (
    df["Private Hospitals"] +
    df["Private PCFs"] +
    df["Private Infirmaries"] +
    df["Private Clinical Laboratories"]
)

healthcare_cols = [
    "RHUs",
    "Government Hospitals", "Private Hospitals",
    "Government PCFs", "Private PCFs",
    "Government Infirmaries", "Private Infirmaries",
    "Government Clinical Laboratories", "Private Clinical Laboratories",
    "Total Healthcare Facilities"
]

df.drop(columns=healthcare_cols, inplace=True)


In [14]:
# Dropping Unnecessary Columns
cols_to_drop = [
    "Latitude",
    "Longitude",
    "Score",
    "Total Percentage",
    "Provincial Score",
    "Density",
    "Urban Population 2020",
    "Urban Percent 2020",
    "Date",
    "Storm Surge Hazard Level",
    "Year"
]

df.drop(columns=cols_to_drop, inplace=True)

In [15]:
# Variable Groups

climate_vars = [
    "Rain",
    "Temperature",
    "Relative Humidity"
]

land_cover_vars = [
    "Forest Cover (%)",
    "Agricultural Land (%)",
    "Water Bodies (%)",
    "Open Vegetation (%)",
    "Built-up / Barren (%)"
]

environmental_vars = [
    "Flood Hazard Level",
    "Settlement Type"
] + land_cover_vars

healthcare_vars = [
    "Public Healthcare Facilities",
    "Private Healthcare Facilities"
]

socioeconomic_vars = [
    "Total Population",
    "Municipal Class"
] + healthcare_vars

lagged_vars = ["Dengue Incidence Lag1", "Rain Lag 1", "Temperature Lag 1", "Relative Humidity Lag 1"]

temporal_vars = ["Month"]

spatial_ids = [
    "Province",
    "Municipality"
]

epidemiological_vars = [
    "Dengue Cases",
    "Death Cases"
]

feature_groups = {
    "Climate": climate_vars,
    "Environmental": environmental_vars,
    "Socioeconomic": socioeconomic_vars,
    "Lagged": lagged_vars,
    "Temporal": temporal_vars,
    "Epidemiological": epidemiological_vars
}

### **Descriptive Statistics**

In [16]:
# Descriptive Statistics (Numerical)
def descriptive_numeric(dataframe, cols, group_name):
    cols = [col for col in cols if col in dataframe.columns]
    if not cols:
        return None
    
    numeric_cols = dataframe[cols].select_dtypes(include="number").columns.tolist()
    if not numeric_cols:
        return None

    # Computation 
    desc = dataframe[numeric_cols].describe().T
    desc["variance"] = dataframe[numeric_cols].var(numeric_only=True)
    desc["missing_count"] = dataframe[numeric_cols].isna().sum()
    desc["missing_%"] = (dataframe[numeric_cols].isna().mean() * 100).round(2)

    # Reorder columns
    desc = desc[[
        "count", "mean", "std", "min", "25%", "50%", "75%", "max",
        "variance", "missing_count", "missing_%"
    ]]

    return desc.round(4)

desc_tables = {}
print("\n=== DESCRIPTIVE STATISTICS (NUMERICAL) ===")
for group_name, cols in feature_groups.items():
    table = descriptive_numeric(df, cols, group_name)
    if table is not None:
        desc_tables[group_name] = table
        print(f"\n--- {group_name} (Numeric) ---")
        print(table)


=== DESCRIPTIVE STATISTICS (NUMERICAL) ===

--- Climate (Numeric) ---
                    count      mean       std      min      25%       50%  \
Rain               8103.0  210.7040  155.3960   0.3000  80.6000  196.3000   
Temperature        8103.0   26.6345    1.4621  20.7328  25.7346   26.8262   
Relative Humidity  8103.0   82.2420    5.8204  59.8344  79.0186   83.4669   

                        75%        max    variance  missing_count  missing_%  
Rain               302.0500  1186.3000  24147.9321              0        0.0  
Temperature         27.5877    30.8598      2.1379              0        0.0  
Relative Humidity   86.4792    97.4132     33.8770              0        0.0  

--- Environmental (Numeric) ---
                        count     mean      std  min    25%    50%    75%  \
Forest Cover (%)       8103.0   7.0416  13.2262  0.0   0.00   1.00   8.02   
Agricultural Land (%)  8103.0  54.7971  28.9939  0.0  30.73  60.14  79.71   
Water Bodies (%)       8103.0   1.9460  

In [17]:
# Descriptive Statistics (Categorical)
categorical_candidates = (
    spatial_ids + temporal_vars + [
        "Settlement Type",
        "Municipal Class",
        "Dengue Risk Level",
        "Flood Hazard Level"
    ]
)

cat_tables = {}
print("\n=== DESCRIPTIVE STATISTICS (CATEGORICAL) ===")
for col in categorical_candidates:
    if col in df.columns:  # check column exists
        if df[col].dtype == "object" or str(df[col].dtype).startswith("category"):
            freq = df[col].value_counts(dropna=False)
            pct = (df[col].value_counts(normalize=True, dropna=False) * 100).round(2)
            cat_tables[col] = pd.DataFrame({"Frequency": freq, "Percent (%)": pct})

            print(f"\n--- {col} ---")
            print(cat_tables[col])


=== DESCRIPTIVE STATISTICS (CATEGORICAL) ===

--- Province ---
          Frequency  Percent (%)
Province                        
QUEZON         2203        27.19
BATANGAS       1947        24.03
LAGUNA         1770        21.84
CAVITE         1357        16.75
RIZAL           826        10.19

--- Municipality ---
              Frequency  Percent (%)
Municipality                        
ROSARIO             118         1.46
AGONCILLO            59         0.73
GUMACA               59         0.73
DOLORES              59         0.73
CATANAUAN            59         0.73
...                 ...          ...
TAGKAWAYAN           47         0.58
SAN NARCISO          47         0.58
SAMPALOC             47         0.58
JOMALIG              23         0.28
PATNANUNGAN          23         0.28

[140 rows x 2 columns]

--- Settlement Type ---
                 Frequency  Percent (%)
Settlement Type                        
URBAN                 6771        83.56
RURAL                 1332       

In [18]:
# Regional Variation Summaries 
# Dengue Cases & Death Cases by Province
for var in ["Dengue Cases", "Death Cases"]:
    if "Province" in df.columns and var in df.columns:
        prov_summary = (
            df.groupby("Province")[var]
              .agg(["count", "sum", "mean", "std", "min", "max"])
              .sort_values("sum", ascending=False)
              .round(4)
        )
        print(f"\n=== {var.upper()} SUMMARY BY PROVINCE ===")
        print(prov_summary)

# Dengue Cases by Month (Seasonality)
if "Month" in df.columns and "Dengue Cases" in df.columns:
    month_summary = (
        df.groupby("Month")["Dengue Cases"]
          .agg(["count", "sum", "mean", "std", "min", "max"])
          .sort_index()
          .round(4)
    )
    print("\n=== DENGUE CASES SUMMARY BY MONTH ===")
    print(month_summary)


=== DENGUE CASES SUMMARY BY PROVINCE ===
          count    sum     mean      std  min  max
Province                                          
RIZAL       826  30111  36.4540  79.2238    0  672
CAVITE     1357  29498  21.7377  53.0633    0  555
LAGUNA     1770  23960  13.5367  34.5457    0  386
QUEZON     2203  16193   7.3504  17.5104    0  269
BATANGAS   1947  11804   6.0627  15.6416    0  208

=== DEATH CASES SUMMARY BY PROVINCE ===
          count  sum    mean     std  min  max
Province                                      
CAVITE     1357   94  0.0693  0.3138    0    4
RIZAL       826   72  0.0872  0.3406    0    3
LAGUNA     1770   71  0.0401  0.2155    0    2
QUEZON     2203   46  0.0209  0.1552    0    2
BATANGAS   1947   37  0.0190  0.1403    0    2

=== DENGUE CASES SUMMARY BY MONTH ===
       count    sum     mean      std  min  max
Month                                          
1        546   5253   9.6209  22.5516    0  325
2        687   5867   8.5400  18.3395    0  243


### **Correlation Analysis**

In [19]:
# Variable Selection
corr_vars = (
    epidemiological_vars +   
    climate_vars +           
    land_cover_vars +
    healthcare_vars +     
    [
        "Total Population",
    ] + lagged_vars
)

corr_vars = [
    c for c in corr_vars
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]

# Remove duplicates (safe guard)
corr_vars = list(dict.fromkeys(corr_vars))

# Normality Test (Shapiro-Wilk)
normality_results = []

for col in corr_vars:
    stat, p = shapiro(df[col].dropna().sample(500, random_state=42))  
    normality_results.append({
        "Variable": col,
        "Shapiro_p_value": round(p, 5),
        "Normality": "Normal" if p > 0.05 else "Non-normal"
    })

normality_df = pd.DataFrame(normality_results)
print("\n=== NORMALITY TEST (Shapiro–Wilk) ===")
print(normality_df)

if all(normality_df["Normality"] == "Non-normal"):
    print("All variables are non-normal. Proceeding with Spearman correlation analysis.\n")
else:
    print("Some variables are normal, consider Pearson correlation for those.\n")


=== NORMALITY TEST (Shapiro–Wilk) ===
                         Variable  Shapiro_p_value   Normality
0                    Dengue Cases          0.00000  Non-normal
1                     Death Cases          0.00000  Non-normal
2                            Rain          0.00000  Non-normal
3                     Temperature          0.00014  Non-normal
4               Relative Humidity          0.00000  Non-normal
5                Forest Cover (%)          0.00000  Non-normal
6           Agricultural Land (%)          0.00000  Non-normal
7                Water Bodies (%)          0.00000  Non-normal
8             Open Vegetation (%)          0.00000  Non-normal
9           Built-up / Barren (%)          0.00000  Non-normal
10   Public Healthcare Facilities          0.00000  Non-normal
11  Private Healthcare Facilities          0.00000  Non-normal
12               Total Population          0.00000  Non-normal
13                     Rain Lag 1          0.00000  Non-normal
14              

In [20]:
# Spearman Analysis
outcome_var = "Dengue Incidence"
if outcome_var not in corr_vars:
    corr_vars = [outcome_var] + corr_vars

spearman_corr = df[corr_vars].corr(method="spearman")
print("=== SPEARMAN CORRELATION ANALYSIS ===")
print(spearman_corr.round(3))

# Correlation with Dengue Incidence
dengue_corr = spearman_corr[outcome_var].sort_values(ascending=False)
print(f"\n=== SPEARMAN CORRELATION WITH {outcome_var.upper()} ===")
print(dengue_corr.round(3))

# Classify correlation strength
def corr_strength(r):
    r = abs(r)
    if r >= 0.8:
        return "Very Strong"
    elif r >= 0.6:
        return "Strong"
    elif r >= 0.4:
        return "Moderate"
    elif r >= 0.2:
        return "Weak"
    else:
        return "Very Weak"

strength_df = pd.DataFrame({
    "Spearman_r": dengue_corr.round(3),
    "Strength": dengue_corr.apply(corr_strength)
})

print(f"\n=== CORRELATION STRENGTH CLASSIFICATION ({outcome_var}) ===")
print(strength_df)

# Predictor–predictor correlations (exclude outcomes)
predictor_vars = [c for c in corr_vars if c != outcome_var]
predictor_corr = spearman_corr.loc[predictor_vars, predictor_vars]
print("\n=== SPEARMAN CORRELATION BETWEEN PREDICTORS ===")
print(predictor_corr.round(3))

# Identify highly correlated predictor pairs (|r| ≥ 0.7)
high_corr_pairs = []

for i in range(len(predictor_vars)):
    for j in range(i + 1, len(predictor_vars)):
        r = predictor_corr.iloc[i, j]
        if abs(r) >= 0.7:
            high_corr_pairs.append({
                "Variable_1": predictor_vars[i],
                "Variable_2": predictor_vars[j],
                "Spearman_r": round(r, 3)
            })

high_corr_df = pd.DataFrame(high_corr_pairs)
print("\n=== HIGHLY CORRELATED PREDICTOR PAIRS (|r| ≥ 0.7) ===")
print(high_corr_df if not high_corr_df.empty else "None found")

print("\nSpearman correlation analysis completed successfully.")

=== SPEARMAN CORRELATION ANALYSIS ===
                               Dengue Incidence  Dengue Cases  Death Cases  \
Dengue Incidence                          1.000         0.877        0.170   
Dengue Cases                              0.877         1.000        0.219   
Death Cases                               0.170         0.219        1.000   
Rain                                      0.145         0.085        0.040   
Temperature                              -0.071        -0.031        0.001   
Relative Humidity                         0.170         0.080        0.038   
Forest Cover (%)                         -0.001        -0.101       -0.027   
Agricultural Land (%)                    -0.076        -0.234       -0.095   
Water Bodies (%)                          0.040         0.083        0.041   
Open Vegetation (%)                       0.002         0.052        0.032   
Built-up / Barren (%)                     0.125         0.379        0.120   
Public Healthcare Faciliti

### **Analysis of Variance (ANOVA)**

In [21]:
# Variable Selection
anova_vars = (
    climate_vars +           
    land_cover_vars +
    healthcare_vars +     
    ["Total Population"]
)

anova_vars = [v for v in anova_vars if v in df.columns and pd.api.types.is_numeric_dtype(df[v])]
anova_vars = list(dict.fromkeys(anova_vars))

# One-Way ANOVA by Dengue Risk Level

anova_results = []

for var in anova_vars:
    # Split variable by Dengue Risk Level
    groups = [
        df[df["Dengue Risk Level"] == level][var].dropna()
        for level in df["Dengue Risk Level"].unique()
    ]

    # Only run ANOVA if all groups have >1 observation
    if all(len(g) > 1 for g in groups):
        f_stat, p_val = f_oneway(*groups)

        anova_results.append({
            "Variable": var,
            "F_statistic": round(f_stat, 4),
            "p_value": round(p_val, 6),
            "Significant (α=0.05)": "Yes" if p_val < 0.05 else "No"
        })

# Results
anova_df = pd.DataFrame(anova_results)
print("\n=== ONE-WAY ANOVA RESULTS (BY DENGUE RISK LEVEL) ===")
print(anova_df)
print("\n ANOVA analysis completed successfully.")


=== ONE-WAY ANOVA RESULTS (BY DENGUE RISK LEVEL) ===
                         Variable  F_statistic   p_value Significant (α=0.05)
0                            Rain     111.8166  0.000000                  Yes
1                     Temperature       9.6464  0.000065                  Yes
2               Relative Humidity     154.5497  0.000000                  Yes
3                Forest Cover (%)       2.0727  0.125914                   No
4           Agricultural Land (%)       1.7828  0.168233                   No
5                Water Bodies (%)       2.3545  0.095006                   No
6             Open Vegetation (%)       1.1658  0.311711                   No
7           Built-up / Barren (%)       3.5597  0.028491                  Yes
8    Public Healthcare Facilities      11.7027  0.000008                  Yes
9   Private Healthcare Facilities       8.7759  0.000156                  Yes
10               Total Population      14.5870  0.000000                  Yes

 ANOVA an

In [22]:
# Mean values of numeric predictors by dengue risk level
group_means = (
    df.groupby("Dengue Risk Level")[anova_vars]
      .mean()
      .round(3)
)

print("\n=== MEAN VALUES BY DENGUE RISK LEVEL ===")
print(group_means)


=== MEAN VALUES BY DENGUE RISK LEVEL ===
                      Rain  Temperature  Relative Humidity  Forest Cover (%)  \
Dengue Risk Level                                                              
High               283.653       26.730             84.996             7.342   
Low                195.887       26.667             81.564             6.856   
Medium             238.972       26.505             83.690             7.556   

                   Agricultural Land (%)  Water Bodies (%)  \
Dengue Risk Level                                            
High                              53.858             1.732   
Low                               55.179             1.922   
Medium                            53.822             2.078   

                   Open Vegetation (%)  Built-up / Barren (%)  \
Dengue Risk Level                                               
High                            18.027                 19.042   
Low                             18.806             

### **Chi-Square Test of Independence**

In [23]:
# Variable Selection
chi_vars = [
    "Flood Hazard Level",
    "Municipal Class",
    "Settlement Type"
]

chi_vars = [v for v in chi_vars if v in df.columns]

# Chi-Square TEst + Cramer's V
chi_results = []
cramers_results = []

for var in chi_vars:
    # Contingency table with Dengue_Risk_Level
    contingency = pd.crosstab(df[var], df["Dengue Risk Level"])

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        print(f"Skipped {var}: insufficient categories")
        continue

    # Chi-Square test
    chi2, p, dof, expected = chi2_contingency(contingency)

    chi_results.append({
        "Variable": var,
        "Chi2_statistic": round(chi2, 4),
        "Degrees_of_Freedom": dof,
        "p_value": round(p, 6),
        "Significant (α=0.05)": "Yes" if p < 0.05 else "No"
    })

    # Cramér's V effect size
    n = contingency.values.sum()
    r, k = contingency.shape
    v = np.sqrt(chi2 / (n * (min(r, k) - 1)))

    # Association strength
    if v < 0.10:
        strength = "Weak"
    elif v < 0.30:
        strength = "Moderate"
    else:
        strength = "Strong"

    cramers_results.append({
        "Variable": var,
        "Cramers_V": round(v, 4),
        "Association_Strength": strength
    })

    print(f"\n=== CONTINGENCY TABLE: {var} vs Dengue_Risk_Level ===")
    print(contingency)

# Results
chi_df = pd.DataFrame(chi_results)
cramers_df = pd.DataFrame(cramers_results)

print("\n=== CHI-SQUARE TEST RESULTS ===")
print(chi_df)

print("\n=== CRAMÉR'S V EFFECT SIZE ===")
print(cramers_df)

print("\nChi-Square analysis completed successfully.")


=== CONTINGENCY TABLE: Flood Hazard Level vs Dengue_Risk_Level ===
Dengue Risk Level   High   Low  Medium
Flood Hazard Level                    
HIGH                  29   425     136
LITTLE TO NONE       304  3684    1177
LOW                   50   797     215
MEDIUM                92   903     291

=== CONTINGENCY TABLE: Municipal Class vs Dengue_Risk_Level ===
Dengue Risk Level          High   Low  Medium
Municipal Class                              
Component City               85   814     281
Fifth Class Municipality     81   636     190
First Class Municipality    146  1781     562
Fourth Class Municipality    85  1198     380
Highly Urbanized City         2    39      18
Second Class Municipality    39   527     201
Third Class Municipality     37   814     187

=== CONTINGENCY TABLE: Settlement Type vs Dengue_Risk_Level ===
Dengue Risk Level  High   Low  Medium
Settlement Type                      
RURAL               102   952     278
URBAN               373  4857    1541

=

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8103 entries, 3181 to 8243
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Province                       8103 non-null   object 
 1   Municipality                   8103 non-null   object 
 2   Month                          8103 non-null   int64  
 3   Dengue Cases                   8103 non-null   int64  
 4   Death Cases                    8103 non-null   int64  
 5   Rain                           8103 non-null   float64
 6   Temperature                    8103 non-null   float64
 7   Relative Humidity              8103 non-null   float64
 8   Municipal Class                8103 non-null   object 
 9   Flood Hazard Level             8103 non-null   object 
 10  Total Population               8103 non-null   float64
 11  Settlement Type                8103 non-null   object 
 12  Dengue Incidence               8103 non-null   flo

In [25]:
# Save
# df.to_excel("Thesis_Dataset_Processed.xlsx", index=False)